# Anisotropic network model

Use a 12-angstrom contact cutoff and the explicit NumPy engine. Physical
`stiffness` calibration is unavailable; keep `stiffness=None`.

In [ ]:
from importlib.resources import as_file, files
import numpy as np
import molsysmt as msm
from elastnetmt import AnisotropicNetworkModel, GaussianNetworkModel, pyunitwizard as puw

resource = files("molsysmt").joinpath("data/pdb/1tcd.pdb")
with as_file(resource) as path:
    molecular_system = msm.convert(str(path), to_form="molsysmt.MolSys")

In [ ]:
anm = AnisotropicNetworkModel(molecular_system, engine="vectorized")
values = anm.get_eigenvalues()
all_values = anm.get_eigenvalues(include_rigid_modes=True)
modes = anm.get_modes()
assert modes.shape == (3 * anm.n_nodes - 6, anm.n_nodes, 3)
np.testing.assert_array_equal(all_values[:6], np.zeros(6))
np.testing.assert_array_equal(values, all_values[6:])
assert np.all(values > 0)

## A trajectory with physical amplitude

Mode indexing is zero-based after excluding the six rigid motions. Target
selection is independent of node selection. Eight frames sample the sine
extrema, so the largest target displacement reaches the requested amplitude.

In [ ]:
trajectory = anm.trajectory_along_mode(
    mode=0, selection='atom_name=="CA"', amplitude="2 angstroms",
    oscillation_steps=8, interpolation_engine="vectorized",
)
original = puw.get_value(
    msm.get(anm.molecular_system, selection=anm.atom_indices, coordinates=True),
    to_unit="nanometers",
)[0]
coordinates = puw.get_value(msm.get(trajectory, coordinates=True), to_unit="nanometers")
assert coordinates.shape == (8, anm.n_nodes, 3)
displacements = np.linalg.norm(coordinates - original, axis=-1)
np.testing.assert_allclose(displacements.max(), 0.2, atol=1e-10)
np.testing.assert_allclose(coordinates[0], original, atol=1e-10)

Connected but geometrically underconstrained networks raise
`DegenerateNetworkError`. Eigenvector signs and bases inside degenerate
subspaces may differ between solvers. Compare residuals, orthogonality and
subspaces when checking numerical parity. See the [API](../../../../api/users/models.rst).